## 6. Save for Ensemble

✅ Gaussian Process predictions ready for ensemble combination

**Output Summary:**
- Predictions shape: (96,) - matches test set
- Scale: [0, 1] range (scaled)
- Format: Numpy array
- Metric (R²): {:.4f}

These predictions will be combined with LSTM, RNN, and Random Forest predictions in the ensemble notebook.

**Next:** Run the Ensemble notebook to combine all 4 model predictions into a unified forecast.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Standard Deviation Distribution
axes[0].hist(gp_std, bins=30, color='#2E86AB', alpha=0.7, edgecolor='black')
axes[0].axvline(gp_std.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {gp_std.mean():.4f}')
axes[0].set_title('Prediction Uncertainty Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Standard Deviation')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Uncertainty vs Prediction Error
abs_errors = np.abs(residuals)
axes[1].scatter(gp_std, abs_errors, alpha=0.6, s=30, color='#A23B72', edgecolors='black', linewidth=0.5)
axes[1].set_title('Uncertainty vs Prediction Error', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Standard Deviation')
axes[1].set_ylabel('Absolute Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✓ Uncertainty range: [{gp_std.min():.4f}, {gp_std.max():.4f}]")

## 5. Uncertainty Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Actual vs Predicted with confidence interval (first 100 samples)
x_range = np.arange(100)
axes[0].plot(x_range, y_test_flat[:100], label='Actual', linewidth=2.5, color='#2E86AB', zorder=3)
axes[0].plot(x_range, gp_predictions_scaled[:100], label='GP Predicted', linewidth=2.5, 
             color='#A23B72', zorder=3)

# Add confidence bands
axes[0].fill_between(x_range, 
                      gp_predictions_scaled[:100] - gp_std[:100],
                      gp_predictions_scaled[:100] + gp_std[:100],
                      alpha=0.3, color='#A23B72', label='Uncertainty ±1σ')

axes[0].set_title('Gaussian Process: Actual vs Predicted with Uncertainty (First 100 samples)', 
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Scaled Demand')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_test_flat - gp_predictions_scaled
axes[1].plot(residuals, label='Residuals', linewidth=1, alpha=0.7, color='#F18F01')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].fill_between(range(len(residuals)), residuals, alpha=0.3, color='#F18F01')
axes[1].set_title('Gaussian Process: Prediction Residuals', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction visualizations created")

## 4. Visualize Predictions with Confidence Bands

In [ ]:
# Generate predictions with uncertainty estimates
gp_predictions_scaled, gp_std = model.predict(X_test_flat, return_std=True)

# Inverse scale to original range
gp_predictions = preprocessing.inverse_scale_predictions(gp_predictions_scaled, scaler_y)

print(f"Gaussian Process Predictions shape: {gp_predictions.shape}")
print(f"Gaussian Process Predictions range: [{gp_predictions.min():.2f}, {gp_predictions.max():.2f}]")
print(f"\nFirst 10 predictions: {gp_predictions[:10]}")
print(f"First 10 standard deviations: {gp_std[:10]}")

# Calculate metrics
gp_mse = mean_squared_error(y_test_flat, gp_predictions_scaled)
gp_mae = mean_absolute_error(y_test_flat, gp_predictions_scaled)
gp_r2 = r2_score(y_test_flat, gp_predictions_scaled)

print(f"\n📈 Gaussian Process Model Performance (on scaled data):")
print(f"  MSE: {gp_mse:.4f}")
print(f"  MAE: {gp_mae:.4f}")
print(f"  R²:  {gp_r2:.4f}")
print(f"  Mean Std: {gp_std.mean():.4f}")

## 3. Generate Predictions and Uncertainty

In [ ]:
print("🚀 Training Gaussian Process model...")

# Define kernel
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# Create and train Gaussian Process
model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    random_state=42,
    n_restarts_optimizer=10
)

model.fit(X_train_flat, y_train_flat)

print("✓ Training completed")
print(f"✓ Kernel: {model.kernel_}")
print(f"✓ Normalized Y: {model.normalize_y}")

## 2. Build and Train Gaussian Process

In [ ]:
# Load data (SAME as all other models)
X, y = preprocessing.load_synthetic_data(n_samples=500, n_features=10)

# Unified preprocessing
data = preprocessing.preprocess_data(X, y, test_size=0.2, seq_length=10)

# Extract FLATTENED data for Gaussian Process
X_train_flat = data['X_train_flat']
X_test_flat = data['X_test_flat']
y_train_flat = data['y_train_flat']
y_test_flat = data['y_test_flat']
scaler_y = data['scaler_y']

print(f"Training data shapes (for Gaussian Process):")
print(f"  X_train: {X_train_flat.shape} (samples, features) - FLATTENED")
print(f"  y_train: {y_train_flat.shape}")
print(f"\nTest data shapes (for Gaussian Process):")
print(f"  X_test: {X_test_flat.shape}")
print(f"  y_test: {y_test_flat.shape}")

# IMPORTANT: Verify output dimension
print(f"\n✓ Test predictions will have shape: ({len(y_test_flat)},)")

## 1. Load and Preprocess Data

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src import preprocessing
from src.models import gaussian_model

# Set random seed
np.random.seed(42)

print("✓ Libraries and modules imported successfully")

# Demand Forecasting System - Gaussian Process Model

## Overview
This notebook trains a **Gaussian Process Regression** model for demand forecasting using scikit-learn.

**Key Features:**
- Input shape: (n_samples, n_features) - flattened
- Output shape: (n_test_samples,) - **MUST match other models**
- RBF (Radial Basis Function) kernel
- Output: predictions in [0, 1] range (scaled)

**Critical:** All predictions must have the **same shape and scale** for ensemble averaging.